# PART IV: Clustering

In [ ]:
import sklearn as sk
import pandas as pd

Clustering Techniques: Use the scikit learn clustering library: https://scikit-learn.org/stable/modules/clustering.htmlLinks to an external site. Note that this page contains very useful illustrations and overviews of the clustering methods, worth reading!
K-means clustering: cluster.KMeansLinks to an external site..
Hierarchical clustering: Experiment with "ward", "complete", "average", and "single" linkage: cluster.AgglomerativeClusteringLinks to an external site..
DBSCAN clustering: Run experiments to find good parameter values. cluster.DBSCANLinks to an external site..

## Importing data

In [ ]:
df = pd.read_parquet("advanced_processed_data.parquet")

## Data Pre-processing

Scale all continuous attributes to a 0 to 1 scale using MinMaxScaler

In [ ]:
from sklearn.preprocessing import MinMaxScaler

numeric_columns = df.select_dtypes(include="number").columns

df[numeric_columns] = MinMaxScaler().fit_transform(df[numeric_columns])

Take 10% subset

In [ ]:
from sklearn.model_selection import train_test_split

sample_df, _ = train_test_split(
    df,
    train_size=0.1,
    stratify=df["Severity"],
    random_state=42,
)

X = sample_df.drop(columns=["Severity"])
y = sample_df["Severity"]

## Clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.feature_selection import f_classif
import matplotlib.pyplot as plt
import numpy as np

def make_cluster_plot(model, X):
    scores = np.zeros(X.shape[1])
    varying_columns = X.columns[X.var(axis=0) > 0]
    if len(varying_columns):
        varying_scores, _ = f_classif(X[varying_columns], model.labels_)
        scores[X.columns.get_indexer(varying_columns)] = varying_scores

    finite_scores = scores[np.isfinite(scores) & (scores > 0)]
    score_cap = finite_scores.max() if finite_scores.size else 1.0
    scores = np.nan_to_num(scores, nan=0.0, posinf=score_cap, neginf=0.0)
    top_positions = np.argsort(scores)[-10:]

    plt.figure(figsize=(9, max(4, 0.45 * len(top_positions))))
    plt.barh(
        X.columns[top_positions],
        scores[top_positions],
        color="teal",
    )
    plt.xlabel("ANOVA F-score (higher = stronger cluster separation)")
    plt.ylabel("Feature")
    plt.title("Features most associated with cluster assignments")
    plt.tight_layout()
    return plt

### K-means Clustering

In [ ]:
n2 = KMeans(n_clusters=2, random_state=0).fit(X)
make_cluster_plot(n2, X).show()

In [ ]:
n20 = KMeans(n_clusters=50, random_state=0).fit(X)
make_cluster_plot(n20, X).show()

In [ ]:
n200 = KMeans(n_clusters=200, random_state=0).fit(X)
make_cluster_plot(n200, X).show()